# LeetCode 76 — Minimum Window Substring (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minWindow(self, s: str, t: str) -> str:

```

Đây là bài "kinh điển nhất" của kỹ thuật **sliding window** (cửa sổ trượt) với kích thước **thay đổi**. Rất nhiều bài Medium/Hard khác (Find All Anagrams, Permutation in String, Longest Substring Without Repeating Characters, ...) chỉ là "anh em họ" của bài này. Học kỹ bài này là bạn có trong tay một **khuôn mẫu** dùng được cho cả một họ bài.

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: "cửa sổ chứa mọi ký tự của `t`, **kể cả lặp**" nghĩa là gì.
2. Nền tảng Python: chuỗi con bằng slicing `s[i:j]`, đếm tần suất bằng `dict` và `collections.Counter`, kiểm tra "bảng đếm A chứa bảng đếm B".
3. Bốn cách giải đi từ dễ đến tối ưu:
   - **Cách 1:** Brute force — thử mọi chuỗi con, đếm lại từ đầu: `O(m³)`.
   - **Cách 2:** Cố định đầu trái, mở rộng phải đến khi đủ — đếm dần, dừng sớm: `O(m² · |Σ|)`.
   - **Cách 3 (cách chính):** **Sliding window hai con trỏ** với bảng `need` và biến `formed` (hoặc `missing`): `O(m + n)`.
   - **Cách 4:** Tối ưu hằng số bằng **lọc trước** các vị trí có ký tự thuộc `t` (`filtered_s`).
4. Trace từng bước **mở rộng** / **co** cửa sổ, thấy rõ khi nào `formed` tăng, khi nào giảm.
5. Vì sao cách 3 đúng (invariant), độ phức tạp, lỗi hay gặp, cách nhận dạng pattern và các bài liên quan.

Ký hiệu dùng xuyên suốt (theo đúng LeetCode): `m = len(s)`, `n = len(t)`, `|Σ|` = số ký tự khác nhau có thể xuất hiện (ở bài này là 52 chữ cái hoa + thường).

## 1. Đề bài (theo LeetCode)

Given two strings `s` and `t` of lengths `m` and `n` respectively, return *the **minimum window substring** of* `s` *such that every character in* `t` *(**including duplicates**) is included in the window*. If there is no such substring, return *the empty string* `""`.

The testcases will be generated such that the answer is **unique**.

### Dịch nhanh

- Cho hai chuỗi `s` (dài `m`) và `t` (dài `n`).
- Tìm **chuỗi con liên tiếp ngắn nhất** của `s` (gọi là **cửa sổ** — *window*) sao cho **mọi ký tự của `t` đều có mặt trong cửa sổ, tính cả số lần lặp**.
  - Ví dụ `t = "AAB"` thì cửa sổ phải có **ít nhất 2 chữ `A`** và **ít nhất 1 chữ `B`**.
  - Thứ tự ký tự **không** quan trọng, cũng **không** cần đứng sát nhau; cửa sổ được phép chứa thêm ký tự "rác".
- Không có cửa sổ nào như vậy ⇒ trả về chuỗi rỗng `""`.
- Đề đảm bảo đáp án (nếu có) là **duy nhất**.

### Ví dụ trong đề

**Example 1**

```text
Input:  s = "ADOBECODEBANC", t = "ABC"
Output: "BANC"
Explanation: The minimum window substring "BANC" includes 'A', 'B', and 'C' from string t.
```

**Example 2**

```text
Input:  s = "a", t = "a"
Output: "a"
Explanation: The entire string s is the minimum window.
```

**Example 3**

```text
Input:  s = "a", t = "aa"
Output: ""
Explanation: Both 'a's from t must be included in the window.
Since the largest window of s only has one 'a', return empty string.
```

### Constraints

- `m == s.length`
- `n == t.length`
- `1 <= m, n <= 10^5`
- `s` and `t` consist of uppercase and lowercase English letters.

**Follow up:** Could you find an algorithm that runs in `O(m + n)` time?

Hai chi tiết quan trọng từ constraints:

1. `m, n` tới `10^5` ⇒ cách `O(m²)` là khoảng `10^10` phép tính — **chắc chắn quá thời gian**. Đề đã gợi ý luôn mục tiêu `O(m + n)`.
2. Chữ **hoa** và chữ **thường** là **khác nhau**: `'a' != 'A'`.

## 2. Hiểu đề bằng tay

Lấy Example 1: `s = "ADOBECODEBANC"`, `t = "ABC"`. Ta đánh số vị trí (index) từ `0`:

```text
index:  0 1 2 3 4 5 6 7 8 9 10 11 12
s    :  A D O B E C O D E B A  N  C
```

Thử vài cửa sổ:

| Cửa sổ | Vị trí | Có `A`? | Có `B`? | Có `C`? | Hợp lệ? | Độ dài |
|---|---|:-:|:-:|:-:|:-:|:-:|
| `"ADOBEC"` | `0..5` | ✓ | ✓ | ✓ | ✓ | 6 |
| `"ADOB"` | `0..3` | ✓ | ✓ | ✗ | ✗ | 4 |
| `"CODEBA"` | `5..10` | ✓ | ✓ | ✓ | ✓ | 6 |
| `"EBANC"` | `8..12` | ✓ | ✓ | ✓ | ✓ | 5 |
| `"BANC"` | `9..12` | ✓ | ✓ | ✓ | ✓ | **4** |
| `"ANC"` | `10..12` | ✓ | ✗ | ✓ | ✗ | 3 |

Cửa sổ hợp lệ ngắn nhất là `"BANC"`. Ta kiểm chứng bằng code nhỏ (chưa cần thuật toán gì):

In [1]:
s = "ADOBECODEBANC"
t = "ABC"

for i, j in [(0, 5), (0, 3), (5, 10), (8, 12), (9, 12), (10, 12)]:
    window = s[i:j + 1]                      # chuỗi con từ i tới j (bao gồm j)
    ok = all(ch in window for ch in t)       # đủ A, B, C chưa? (tạm chưa xét lặp)
    print(f"{i:>2}..{j:<2} {window:<8} hợp lệ={ok}  dài={len(window)}")

 0..5  ADOBEC   hợp lệ=True  dài=6
 0..3  ADOB     hợp lệ=False  dài=4
 5..10 CODEBA   hợp lệ=True  dài=6
 8..12 EBANC    hợp lệ=True  dài=5
 9..12 BANC     hợp lệ=True  dài=4
10..12 ANC      hợp lệ=False  dài=3


Nhận xét:

- Cửa sổ là **đoạn liên tiếp** `s[i..j]`, có thể chứa ký tự "rác" (`D`, `O`, `E`, `N`) — không sao cả.
- Câu `ch in window` ở trên **chưa đủ** cho đề này, vì nó bỏ qua số lần lặp. Với `t = "AA"`, cửa sổ `"A"` có `'A' in "A"` là `True` nhưng thực ra **thiếu một chữ `A`**. Example 3 chính là bẫy này.
- Vậy ta cần một công cụ **đếm số lần xuất hiện** của từng ký tự. Đó là nội dung phần nền tảng tiếp theo.

## 3. Nền tảng Python

### 3.1. Chuỗi con và slicing `s[i:j]`

- `s[i]` lấy **một** ký tự ở vị trí `i`.
- `s[i:j]` lấy chuỗi con từ vị trí `i` **đến trước** vị trí `j` (không lấy `j`). Độ dài là `j - i`.
- Muốn lấy đoạn `i..j` **bao gồm** cả `j` thì viết `s[i:j + 1]`, độ dài `j - i + 1`.

Cặp công thức "**bao gồm hai đầu** ⇒ độ dài `right - left + 1`" sẽ xuất hiện liên tục trong sliding window. Nhầm `+1` là lỗi "lệch một" (*off-by-one*) kinh điển.

Một chuỗi dài `m` có bao nhiêu chuỗi con khác rỗng? Chọn đầu trái `i` và đầu phải `j >= i` ⇒ `m + (m-1) + ... + 1 = m(m+1)/2` cách. Với `m = 10^5` là khoảng `5 · 10^9` — con số này giải thích vì sao brute force sẽ thua.

In [2]:
s = "ADOBECODEBANC"
print(s[9])            # 'B'
print(s[9:13])         # 'BANC'  (vị trí 9, 10, 11, 12)
left, right = 9, 12
print(s[left:right + 1], "dài", right - left + 1)

m = len(s)
print("số chuỗi con của s:", m * (m + 1) // 2)
print("với m = 10^5      :", 10**5 * (10**5 + 1) // 2)

B
BANC
BANC dài 4
số chuỗi con của s: 91
với m = 10^5      : 5000050000


### 3.2. Đếm tần suất bằng `dict`

`dict` (từ điển) lưu cặp `key -> value`. Ở đây key là **ký tự**, value là **số lần xuất hiện**.

Cách đếm thủ công: duyệt từng ký tự, nếu chưa có trong dict thì khởi tạo `0`, rồi cộng `1`. Phương thức `d.get(key, 0)` trả về `d[key]` nếu có, ngược lại trả về `0` — giúp viết gọn trong một dòng.

In [3]:
t = "AABC"
count = {}
for ch in t:
    count[ch] = count.get(ch, 0) + 1
print(count)                 # {'A': 2, 'B': 1, 'C': 1}
print(count["A"])            # 2
print(count.get("Z", 0))     # 0 — không có 'Z' nhưng không bị lỗi

{'A': 2, 'B': 1, 'C': 1}
2
0


### 3.3. `collections.Counter` — bảng đếm có sẵn

`Counter` là một `dict` đặc biệt trong thư viện chuẩn, sinh ra để đếm:

- `Counter("AABC")` ⇒ đếm luôn cả chuỗi.
- Truy cập key **chưa có** trả về `0` thay vì báo lỗi `KeyError` ⇒ viết `window[ch] += 1` thoải mái.
- `len(counter)` = số **ký tự khác nhau** (số key), **không phải** tổng số ký tự.

Chú ý tinh tế: nếu bạn giảm một key về `0`, key đó **vẫn còn** trong Counter (`'A' in c` vẫn là `True`). Vì vậy khi cần hỏi "ký tự này có thuộc `t` không", ta hỏi trên bảng `need` của `t` — bảng mà ta **không bao giờ sửa**.

In [4]:
from collections import Counter

need = Counter("AABC")
print(need)                  # Counter({'A': 2, 'B': 1, 'C': 1})
print(len(need))             # 3 ký tự khác nhau (A, B, C) — không phải 4
print(need["Z"])             # 0, không lỗi

window = Counter()
for ch in "BANANA":
    window[ch] += 1          # key chưa có vẫn cộng được
print(window)                # Counter({'A': 3, 'N': 2, 'B': 1})

window["B"] -= 1
print(window["B"], "B" in window)   # 0 True  <- key vẫn còn dù đếm = 0

Counter({'A': 2, 'B': 1, 'C': 1})
3
0
Counter({'A': 3, 'N': 2, 'B': 1})
0 True


### 3.4. Kiểm tra "cửa sổ có chứa đủ `t` không?"

Gọi `need = Counter(t)` (cần bao nhiêu mỗi ký tự) và `window` là bảng đếm của cửa sổ đang xét. Cửa sổ **hợp lệ** khi và chỉ khi:

> với **mọi** ký tự `c` trong `need`: `window[c] >= need[c]`.

Viết bằng Python: `all(window[c] >= k for c, k in need.items())`.

- `need.items()` cho từng cặp `(ký tự, số lượng cần)`.
- `all(...)` trả `True` nếu **mọi** phần tử đều đúng (dừng ngay khi gặp phần tử sai).
- Chi phí: duyệt mọi key của `need` ⇒ `O(|Σ|)` (tối đa 52 key).

Từ Python 3.10, `Counter` còn hỗ trợ phép so sánh `need <= window` với đúng ý nghĩa này ("mọi đếm của `need` đều ≤ đếm tương ứng của `window`"). Trong notebook ta dùng `all(...)` cho rõ ràng và chạy trên mọi phiên bản.

In [5]:
from collections import Counter


def covers(window: Counter, need: Counter) -> bool:
    """True nếu window chứa ĐỦ mọi ký tự của need (tính cả số lần lặp)."""
    return all(window[c] >= k for c, k in need.items())


need = Counter("AABC")
print(covers(Counter("CABA"), need))      # True : A=2, B=1, C=1
print(covers(Counter("CAB"), need))       # False: chỉ 1 chữ A
print(covers(Counter("XAXBXAXC"), need))  # True : rác X không ảnh hưởng
print(covers(Counter("a"), Counter("aa")))  # False — Example 3
print(Counter("AABC") <= Counter("CABA"))   # cùng ý nghĩa (Python >= 3.10)

True
False
True
False
True


### 3.5. Hai chi tiết nhỏ: hoa/thường và `float("inf")`

- `'a'` và `'A'` là **hai ký tự khác nhau** (mã Unicode 97 và 65). Đề nói rõ `s`, `t` gồm chữ hoa và chữ thường, nên **không** được tự ý `.lower()`.
- Khi tìm **giá trị nhỏ nhất**, ta hay khởi tạo `best_len = float("inf")` (vô cực dương). Mọi số thật đều nhỏ hơn nó, nên lần đầu tìm thấy cửa sổ hợp lệ chắc chắn được ghi nhận. Cuối cùng nếu `best_len` vẫn là `inf` ⇒ chưa từng có cửa sổ hợp lệ ⇒ trả `""`.

In [6]:
from collections import Counter

print(ord("a"), ord("A"), "a" == "A")
print(Counter("aA"))                   # hai key riêng biệt

best_len = float("inf")
print(6 < best_len, 10**9 < best_len)  # True True
print(best_len == float("inf"))        # True -> nghĩa là "chưa tìm thấy"

97 65 False
Counter({'a': 1, 'A': 1})
True True
True


## 4. Cách 1 — Brute force: thử mọi chuỗi con

Cách người mới nghĩ ra đầu tiên, rất tự nhiên:

1. Duyệt mọi cặp `(i, j)` với `i <= j`.
2. Lấy chuỗi con `s[i:j+1]`, đếm bằng `Counter` **từ đầu**.
3. Nếu chứa đủ `t` (`covers`) và ngắn hơn đáp án hiện tại ⇒ cập nhật.

Ta chỉ cập nhật khi **ngắn hơn hẳn** (`<`), và duyệt `i` tăng dần, nên nếu có nhiều cửa sổ cùng độ dài nhỏ nhất, hàm trả về cái **nằm bên trái nhất**. (Đề đảm bảo đáp án duy nhất, nhưng quy ước này giúp ta so sánh các cách với nhau trên dữ liệu ngẫu nhiên ở phần test.)

In [7]:
from collections import Counter


def min_window_bruteforce(s: str, t: str) -> str:
    need = Counter(t)
    best = ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            window = Counter(s[i:j + 1])          # đếm lại TỪ ĐẦU: O(j - i + 1)
            if covers(window, need):              # O(|Σ|)
                if best == "" or j - i + 1 < len(best):
                    best = s[i:j + 1]
    return best


print(min_window_bruteforce("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_bruteforce("a", "a"))                 # a
print(repr(min_window_bruteforce("a", "aa")))          # ''

BANC
a
''


### Vì sao cách 1 không dùng được?

- Số cặp `(i, j)`: `m(m+1)/2 ≈ m²/2`.
- Mỗi cặp lại đếm `Counter(s[i:j+1])` từ đầu, tốn `O(j - i + 1)`; tổng độ dài mọi chuỗi con là khoảng `m³/6`.
- Cộng thêm `covers` tốn `O(|Σ|)` mỗi cặp.

⇒ Thời gian `O(m³)` (chính xác hơn `O(m³ + m²·|Σ|)`), bộ nhớ `O(m)` cho chuỗi con tạm. Cùng đo số ký tự phải đếm để "cảm" được tốc độ tăng:

In [8]:
def brute_chars_counted(m: int) -> int:
    # tổng độ dài của mọi chuỗi con s[i..j] — mỗi chuỗi con bị đếm lại từ đầu
    return sum(j - i + 1 for i in range(m) for j in range(i, m))


for m in [10, 100, 200, 400]:
    print(f"m = {m:>4}: số ký tự phải đếm = {brute_chars_counted(m):>12,}")
print("Gấp đôi m -> gấp khoảng 8 lần công việc (đặc trưng của O(m^3)).")
print("Với m = 10^5: khoảng", f"{10**15 // 6:.1e}", "ký tự — vô vọng.")

m =   10: số ký tự phải đếm =          220
m =  100: số ký tự phải đếm =      171,700
m =  200: số ký tự phải đếm =    1,353,400
m =  400: số ký tự phải đếm =   10,746,800
Gấp đôi m -> gấp khoảng 8 lần công việc (đặc trưng của O(m^3)).
Với m = 10^5: khoảng 1.7e+14 ký tự — vô vọng.


## 5. Cách 2 — Cố định đầu trái, mở rộng phải đến khi đủ

Hai chỗ lãng phí của cách 1:

1. **Đếm lại từ đầu.** Chuỗi con `s[i..j+1]` chỉ hơn `s[i..j]` đúng một ký tự `s[j+1]`. Thay vì đếm lại, chỉ cần `window[s[j+1]] += 1`.
2. **Không dừng sớm.** Với một `i` cố định, khi `s[i..j]` đã đủ thì mọi `s[i..j']` với `j' > j` cũng đủ nhưng **dài hơn** ⇒ vô ích. Ta `break` ngay ở `j` đầu tiên đủ.

Vậy với mỗi đầu trái `left`, ta chỉ cần tìm **`right` nhỏ nhất** sao cho `s[left..right]` chứa đủ `t`.

Thêm một quan sát dừng sớm nữa: nếu với một `left` nào đó, mở rộng tới tận cuối `s` vẫn **chưa đủ**, thì mọi `left` lớn hơn (cửa sổ còn ít ký tự hơn) cũng không đủ ⇒ dừng hẳn.

In [9]:
from collections import Counter


def min_window_fix_left(s: str, t: str) -> str:
    need = Counter(t)
    best = ""
    for left in range(len(s)):
        window = Counter()                       # bắt đầu lại cho mỗi left
        for right in range(left, len(s)):
            window[s[right]] += 1                # đếm DẦN, không đếm lại
            if covers(window, need):             # đủ lần đầu -> dừng sớm
                if best == "" or right - left + 1 < len(best):
                    best = s[left:right + 1]
                break
        else:
            # vòng for chạy hết mà không break: tới cuối s vẫn thiếu
            break                                # left lớn hơn cũng vô vọng
    return best


print(min_window_fix_left("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_fix_left("a", "a"))                 # a
print(repr(min_window_fix_left("a", "aa")))          # ''

BANC
a
''


Cú pháp `for ... else` của Python: khối `else` chạy **khi vòng `for` kết thúc bình thường** (không gặp `break`). Ở đây nghĩa là "duyệt hết `right` mà chưa lần nào đủ".

### Trace cách 2 trên Example 1

Với mỗi `left`, in ra `right` đầu tiên làm cửa sổ đủ:

In [10]:
from collections import Counter


def trace_fix_left(s: str, t: str) -> None:
    need = Counter(t)
    best = ""
    for left in range(len(s)):
        window = Counter()
        found = None
        for right in range(left, len(s)):
            window[s[right]] += 1
            if covers(window, need):
                found = right
                break
        if found is None:
            print(f"left={left:>2} ('{s[left]}'): mở tới hết chuỗi vẫn thiếu -> dừng hẳn")
            break
        cand = s[left:found + 1]
        mark = ""
        if best == "" or len(cand) < len(best):
            best = cand
            mark = "  <- tốt nhất mới"
        print(f"left={left:>2} ('{s[left]}'): đủ lần đầu tại right={found:>2} -> {cand!r} (dài {len(cand)}){mark}")
    print("kết quả:", repr(best))


trace_fix_left("ADOBECODEBANC", "ABC")

left= 0 ('A'): đủ lần đầu tại right= 5 -> 'ADOBEC' (dài 6)  <- tốt nhất mới
left= 1 ('D'): đủ lần đầu tại right=10 -> 'DOBECODEBA' (dài 10)
left= 2 ('O'): đủ lần đầu tại right=10 -> 'OBECODEBA' (dài 9)
left= 3 ('B'): đủ lần đầu tại right=10 -> 'BECODEBA' (dài 8)
left= 4 ('E'): đủ lần đầu tại right=10 -> 'ECODEBA' (dài 7)
left= 5 ('C'): đủ lần đầu tại right=10 -> 'CODEBA' (dài 6)
left= 6 ('O'): đủ lần đầu tại right=12 -> 'ODEBANC' (dài 7)
left= 7 ('D'): đủ lần đầu tại right=12 -> 'DEBANC' (dài 6)
left= 8 ('E'): đủ lần đầu tại right=12 -> 'EBANC' (dài 5)  <- tốt nhất mới
left= 9 ('B'): đủ lần đầu tại right=12 -> 'BANC' (dài 4)  <- tốt nhất mới
left=10 ('A'): mở tới hết chuỗi vẫn thiếu -> dừng hẳn
kết quả: 'BANC'


### Cách 2 vẫn chậm ở đâu?

- Mỗi `left` vẫn có thể phải quét `right` tới gần cuối chuỗi ⇒ worst case `O(m²)` lần thêm ký tự, mỗi lần gọi `covers` thêm `O(|Σ|)` ⇒ **`O(m² · |Σ|)`**. Với `m = 10^5` vẫn quá chậm.
- Nhìn kỹ trace: `left = 1, 2, 3, 4, 5` **đều** dừng ở `right = 10`. Mỗi lần tăng `left`, ta **vứt bỏ** `window` rồi đếm lại `D O B E C O D E B A` gần như y hệt. Đây chính là phần việc bị lặp.

Quan sát quan trọng hơn: cột `right` trong trace **không bao giờ giảm** khi `left` tăng: `5, 10, 10, 10, 10, 10, 12, 12, 12, 12`. Ta kiểm chứng bằng code:

In [11]:
from collections import Counter


def first_right_for_each_left(s: str, t: str) -> list:
    need = Counter(t)
    result = []
    for left in range(len(s)):
        window = Counter()
        found = None
        for right in range(left, len(s)):
            window[s[right]] += 1
            if covers(window, need):
                found = right
                break
        result.append(found)
    return result


fr = first_right_for_each_left("ADOBECODEBANC", "ABC")
print(fr)
vals = [x for x in fr if x is not None]
print("không giảm?", all(a <= b for a, b in zip(vals, vals[1:])))

[5, 10, 10, 10, 10, 10, 12, 12, 12, 12, None, None, None]
không giảm? True


## 6. Bước đột phá — Tính đơn điệu (monotonicity)

Vì sao `right` không bao giờ phải lùi lại? Hai sự thật đơn giản:

1. **Thêm ký tự không làm mất tính hợp lệ.** Nếu `s[l..r]` chứa đủ `t` thì `s[l..r+1]` (nhiều hơn một ký tự) cũng chứa đủ `t`. Các đếm chỉ tăng, không giảm.
2. **Bớt ký tự không làm "sửa" được cửa sổ thiếu.** Nếu `s[l..r]` **thiếu** thì `s[l+1..r]` (ít hơn một ký tự) cũng **thiếu**.

Hệ quả: gọi `R(l)` là `right` nhỏ nhất làm `s[l..R(l)]` đủ. Vì `s[l+1..r]` thiếu với mọi `r < R(l)` (do hệ quả 2), nên `R(l+1) >= R(l)`. Tức là **khi `left` tăng, `right` chỉ cần đi tiếp từ chỗ cũ, không bao giờ quay lại**.

Đây là điều kiện vàng để dùng **hai con trỏ cùng chiều** (*two pointers*):

- Con trỏ `right` chạy từ trái sang phải **đúng một lần**.
- Con trỏ `left` cũng chạy từ trái sang phải **đúng một lần**.
- Tổng số bước di chuyển ≤ `2m` ⇒ `O(m)` thay vì `O(m²)`.

Và vì không vứt cửa sổ đi, ta **giữ nguyên** bảng đếm `window`: thêm ký tự khi `right` tiến, bớt ký tự khi `left` tiến. Đó chính là **sliding window**.

## 7. Nền tảng — Sliding window (cửa sổ trượt) qua một bài mini

Trước khi áp vào bài chính, hãy làm quen với khuôn "**mở rộng phải – co trái**" trên một bài nhỏ hơn (LeetCode 209 — Minimum Size Subarray Sum):

> Cho mảng **số dương** `nums` và `target`. Tìm độ dài đoạn con liên tiếp **ngắn nhất** có tổng `>= target`.

Vì các số đều dương, bài này có đúng tính đơn điệu như trên: thêm phần tử thì tổng tăng (vẫn đạt), bớt phần tử thì tổng giảm (đang thiếu thì vẫn thiếu).

Khuôn mẫu:

```text
left = 0
for right in range(len(nums)):        # (1) MỞ RỘNG: thêm nums[right] vào cửa sổ
    cập nhật trạng thái cửa sổ
    while cửa sổ đang HỢP LỆ:         # (2) CO: còn hợp lệ thì cố thu nhỏ
        ghi nhận đáp án (right-left+1)
        bỏ nums[left] khỏi cửa sổ
        left += 1
```

Hai điểm then chốt:

- Ghi nhận đáp án **bên trong** `while`, **trước** khi bỏ `nums[left]` (lúc cửa sổ còn hợp lệ).
- Dùng `while`, không dùng `if`: có thể co được **nhiều** bước liên tiếp.

In [12]:
def min_subarray_len(target: int, nums: list) -> int:
    left = 0
    total = 0
    best = float("inf")
    for right, x in enumerate(nums):
        total += x                                   # (1) mở rộng
        print(f"right={right} thêm {x}: [{left},{right}] sum={total}")
        while total >= target:                       # (2) co khi còn hợp lệ
            best = min(best, right - left + 1)
            print(f"    hợp lệ, dài {right - left + 1} -> best={best}; bỏ {nums[left]}")
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best


print("đáp án:", min_subarray_len(7, [2, 3, 1, 2, 4, 3]))   # 2 ([4, 3])

right=0 thêm 2: [0,0] sum=2
right=1 thêm 3: [0,1] sum=5
right=2 thêm 1: [0,2] sum=6
right=3 thêm 2: [0,3] sum=8
    hợp lệ, dài 4 -> best=4; bỏ 2
right=4 thêm 4: [1,4] sum=10
    hợp lệ, dài 4 -> best=4; bỏ 3
    hợp lệ, dài 3 -> best=3; bỏ 1
right=5 thêm 3: [3,5] sum=9
    hợp lệ, dài 3 -> best=3; bỏ 2
    hợp lệ, dài 2 -> best=2; bỏ 4
đáp án: 2


Đọc output:

- `right=3`: cửa sổ `[2,3,1,2]` tổng 8 ≥ 7 ⇒ ghi nhận 4, bỏ `2` ⇒ tổng 6 < 7, dừng co.
- `right=4`: thêm `4` ⇒ `[3,1,2,4]` tổng 10 ⇒ ghi nhận 4; bỏ `3` ⇒ `[1,2,4]` tổng 7 vẫn hợp lệ ⇒ ghi nhận **3**; bỏ `1` ⇒ tổng 6, dừng. Thấy rõ vì sao phải dùng `while`.
- `right=5`: thêm `3` ⇒ `[2,4,3]` ⇒ 3; bỏ `2` ⇒ `[4,3]` tổng 7 ⇒ **2**; bỏ `4` ⇒ tổng 3, dừng.

Mỗi phần tử được **thêm một lần** (khi `right` đi qua) và **bỏ tối đa một lần** (khi `left` đi qua) ⇒ `O(len(nums))`.

Bài 76 dùng **y hệt** khuôn này. Chỉ khác ở câu hỏi "cửa sổ hợp lệ?" — thay cho `total >= target` là "cửa sổ chứa đủ `t`?". Câu hỏi đó cần được trả lời trong `O(1)`, nếu không mỗi bước lại tốn `O(|Σ|)`.

## 8. Trả lời "đủ chưa?" trong O(1) — biến `formed`

Gọi `required = len(need)` = số **ký tự khác nhau** của `t`. Ta duy trì thêm một con số:

> `formed` = số ký tự khác nhau `c` **đã đủ**, tức là `window[c] >= need[c]`.

Cửa sổ hợp lệ ⇔ `formed == required`. Câu hỏi giờ chỉ là so sánh hai số nguyên — `O(1)`.

Cập nhật `formed` thế nào cho đúng? Mấu chốt là chỉ thay đổi đúng **tại thời điểm "vượt ngưỡng"**:

| Sự kiện | Điều kiện | Cập nhật |
|---|---|---|
| Thêm `c` (right tiến) | sau khi cộng, `window[c] == need[c]` (**vừa** đủ) | `formed += 1` |
| Thêm `c` nhưng đã đủ từ trước | `window[c] > need[c]` (thừa) | không đổi |
| Bỏ `c` (left tiến) | sau khi trừ, `window[c] < need[c]` — chính xác là `== need[c] - 1` (**vừa** hụt) | `formed -= 1` |
| Bỏ `c` nhưng vẫn còn đủ | `window[c] >= need[c]` | không đổi |

Ví dụ số: `t = "AAB"` ⇒ `need = {A: 2, B: 1}`, `required = 2`.

- Thêm `A` lần 1: `window[A] = 1 < 2` ⇒ chưa đủ, `formed = 0`.
- Thêm `A` lần 2: `window[A] = 2 == 2` ⇒ **vừa đủ**, `formed = 1`.
- Thêm `A` lần 3: `window[A] = 3 > 2` ⇒ thừa, `formed` **vẫn là 1** (nếu cộng nữa là sai!).
- Bỏ một `A`: `window[A] = 2` vẫn đủ ⇒ `formed` không đổi.
- Bỏ thêm một `A`: `window[A] = 1 < 2` ⇒ **vừa hụt**, `formed = 0`.

Ký tự không thuộc `t` (như `X`) thì `need[X] = 0`, ta bỏ qua hoàn toàn khi cập nhật `formed`.

In [13]:
from collections import Counter

need = Counter("AAB")
required = len(need)
window = Counter()
formed = 0


def add(ch):
    global formed
    window[ch] += 1
    if ch in need and window[ch] == need[ch]:
        formed += 1
    print(f"thêm {ch}: A={window['A']}/2 B={window['B']}/1 formed={formed}/{required}")


def remove(ch):
    global formed
    window[ch] -= 1
    if ch in need and window[ch] < need[ch]:
        formed -= 1
    print(f"bỏ   {ch}: A={window['A']}/2 B={window['B']}/1 formed={formed}/{required}")


for ch in "AXAAB":
    add(ch)
remove("A")
remove("X")
remove("A")

thêm A: A=1/2 B=0/1 formed=0/2
thêm X: A=1/2 B=0/1 formed=0/2
thêm A: A=2/2 B=0/1 formed=1/2
thêm A: A=3/2 B=0/1 formed=1/2
thêm B: A=3/2 B=1/1 formed=2/2
bỏ   A: A=2/2 B=1/1 formed=2/2
bỏ   X: A=2/2 B=1/1 formed=2/2
bỏ   A: A=1/2 B=1/1 formed=1/2


Đúng như bảng: lần thêm `A` thứ ba **không** làm `formed` tăng; lần bỏ `A` thứ nhất không làm giảm (vẫn còn 2), lần bỏ thứ hai mới làm `formed` tụt từ 2 xuống 1. Ở dòng `thêm B`, `formed` đạt `2/2` ⇒ cửa sổ `"AXAAB"` hợp lệ.

(Trong cell này ta dùng `global` chỉ để demo hai hàm nhỏ dùng chung biến; trong thuật toán thật mọi thứ nằm trong một hàm nên không cần.)

## 9. Cách 3 (cách chính) — Sliding window hai con trỏ với `need` và `formed`

Ghép tất cả lại:

1. `need = Counter(t)`, `required = len(need)`; `window = Counter()`, `formed = 0`, `left = 0`.
2. `best_len = inf`, `best_left = 0` để nhớ cửa sổ tốt nhất (lưu **vị trí + độ dài**, không cắt chuỗi mỗi lần — cắt chuỗi tốn `O(độ dài)`).
3. Với mỗi `right` từ `0` tới `m-1` — **MỞ RỘNG**:
   - `ch = s[right]`, `window[ch] += 1`.
   - Nếu `ch` thuộc `t` và `window[ch] == need[ch]` ⇒ `formed += 1`.
4. `while formed == required` — **CO**:
   - Cửa sổ `[left, right]` đang hợp lệ ⇒ nếu `right - left + 1 < best_len` thì ghi nhận.
   - Bỏ `out = s[left]`: `window[out] -= 1`; nếu `out` thuộc `t` và `window[out] < need[out]` ⇒ `formed -= 1`.
   - `left += 1`.
5. Trả về `""` nếu `best_len` còn là `inf`, ngược lại `s[best_left : best_left + best_len]`.

Ý nghĩa của vòng `while`: với mỗi `right`, ta co `left` **xa nhất có thể** mà vẫn hợp lệ ⇒ tìm được cửa sổ hợp lệ **ngắn nhất kết thúc tại `right`**. Đáp án là cửa sổ ngắn nhất trong tất cả các `right`.

In [14]:
from collections import Counter


def min_window_sliding(s: str, t: str) -> str:
    need = Counter(t)
    required = len(need)          # số ký tự KHÁC NHAU phải đủ
    window = Counter()
    formed = 0
    left = 0
    best_len, best_left = float("inf"), 0

    for right, ch in enumerate(s):
        # (1) MỞ RỘNG: đưa s[right] vào cửa sổ
        window[ch] += 1
        if ch in need and window[ch] == need[ch]:
            formed += 1

        # (2) CO: còn hợp lệ thì ghi nhận rồi thu nhỏ từ trái
        while formed == required:
            if right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            out = s[left]
            window[out] -= 1
            if out in need and window[out] < need[out]:
                formed -= 1
            left += 1

    return "" if best_len == float("inf") else s[best_left:best_left + best_len]


print(min_window_sliding("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_sliding("a", "a"))                 # a
print(repr(min_window_sliding("a", "aa")))          # ''

BANC
a
''


## 10. Mô phỏng (trace) từng bước

### Bảng tính tay cho Example 1: `s = "ADOBECODEBANC"`, `t = "ABC"`

`need = {A:1, B:1, C:1}`, `required = 3`. Cột "cửa sổ" là trạng thái **sau khi** thêm `s[right]`, **trước khi** co.

| `right` | thêm | cửa sổ `[left,right]` | `formed` | Co cửa sổ (`while formed == 3`) |
|:-:|:-:|---|:-:|---|
| 0 | `A` | `[0,0]` `"A"` | 1 | — |
| 1–2 | `D`,`O` | `[0,2]` `"ADO"` | 1 | — (ký tự rác) |
| 3 | `B` | `[0,3]` `"ADOB"` | 2 | — |
| 4 | `E` | `[0,4]` `"ADOBE"` | 2 | — |
| 5 | `C` | `[0,5]` `"ADOBEC"` | **3** | ghi nhận `"ADOBEC"` (6); bỏ `A` ⇒ thiếu `A`, `formed=2`, `left=1` |
| 6–8 | `O`,`D`,`E` | `[1,8]` `"DOBECODE"` | 2 | — |
| 9 | `B` | `[1,9]` `"DOBECODEB"` | 2 | — (`B` thứ hai là **thừa**, `formed` không tăng) |
| 10 | `A` | `[1,10]` `"DOBECODEBA"` | **3** | bỏ `D`,`O`,`B`,`E` (bỏ `B`@3 vẫn còn `B`@9 nên vẫn đủ); tới `"CODEBA"` dài 6 — không ngắn hơn 6; bỏ `C` ⇒ `formed=2`, `left=6` |
| 11 | `N` | `[6,11]` `"ODEBAN"` | 2 | — |
| 12 | `C` | `[6,12]` `"ODEBANC"` | **3** | bỏ `O`,`D` ⇒ `"EBANC"` (5) ghi nhận; bỏ `E` ⇒ `"BANC"` (4) ghi nhận; bỏ `B` ⇒ `formed=2`, `left=10` |

Kết quả: `"BANC"`. Hàm `trace_sliding` dưới đây in đúng các bước đó (kèm số đếm từng ký tự của `t` dạng `có/cần`):

In [15]:
from collections import Counter


def trace_sliding(s: str, t: str) -> str:
    need = Counter(t)
    required = len(need)
    window = Counter()
    formed, left = 0, 0
    best_len, best_left = float("inf"), 0

    def counts():
        return " ".join(f"{c}={window[c]}/{need[c]}" for c in sorted(need))

    for right, ch in enumerate(s):
        window[ch] += 1
        note = ""
        if ch in need and window[ch] == need[ch]:
            formed += 1
            note = f"  '{ch}' vừa đủ"
        print(f"right={right:>2} +{ch} [{left},{right}] {s[left:right + 1]!r:<14} {counts()} formed={formed}/{required}{note}")
        while formed == required:
            length = right - left + 1
            rec = ""
            if length < best_len:
                best_len, best_left = length, left
                rec = " GHI NHẬN"
            out = s[left]
            window[out] -= 1
            lost = ""
            if out in need and window[out] < need[out]:
                formed -= 1
                lost = f" -> thiếu '{out}', formed={formed}"
            print(f"      co: {s[left:right + 1]!r} dài {length}{rec}; bỏ '{out}'{lost}")
            left += 1

    ans = "" if best_len == float("inf") else s[best_left:best_left + best_len]
    print("kết quả:", repr(ans))
    return ans


trace_sliding("ADOBECODEBANC", "ABC");

right= 0 +A [0,0] 'A'            A=1/1 B=0/1 C=0/1 formed=1/3  'A' vừa đủ
right= 1 +D [0,1] 'AD'           A=1/1 B=0/1 C=0/1 formed=1/3
right= 2 +O [0,2] 'ADO'          A=1/1 B=0/1 C=0/1 formed=1/3
right= 3 +B [0,3] 'ADOB'         A=1/1 B=1/1 C=0/1 formed=2/3  'B' vừa đủ
right= 4 +E [0,4] 'ADOBE'        A=1/1 B=1/1 C=0/1 formed=2/3
right= 5 +C [0,5] 'ADOBEC'       A=1/1 B=1/1 C=1/1 formed=3/3  'C' vừa đủ
      co: 'ADOBEC' dài 6 GHI NHẬN; bỏ 'A' -> thiếu 'A', formed=2
right= 6 +O [1,6] 'DOBECO'       A=0/1 B=1/1 C=1/1 formed=2/3
right= 7 +D [1,7] 'DOBECOD'      A=0/1 B=1/1 C=1/1 formed=2/3
right= 8 +E [1,8] 'DOBECODE'     A=0/1 B=1/1 C=1/1 formed=2/3
right= 9 +B [1,9] 'DOBECODEB'    A=0/1 B=2/1 C=1/1 formed=2/3
right=10 +A [1,10] 'DOBECODEBA'   A=1/1 B=2/1 C=1/1 formed=3/3  'A' vừa đủ
      co: 'DOBECODEBA' dài 10; bỏ 'D'
      co: 'OBECODEBA' dài 9; bỏ 'O'
      co: 'BECODEBA' dài 8; bỏ 'B'
      co: 'ECODEBA' dài 7; bỏ 'E'
      co: 'CODEBA' dài 6; bỏ 'C' -> thiếu 'C', formed=2
right

Output khớp từng dòng với bảng. Ba điều nên để ý:

1. **`right=9`**: thêm `B` nhưng `B=2/1` là thừa ⇒ `formed` vẫn `2/3`. Nếu viết `>=` thay cho `==` thì `formed` sẽ bị cộng lần hai — lỗi kinh điển (xem phần lỗi thường gặp).
2. **`right=10`**: vòng `while` co **5 bước liên tiếp**. Việc bỏ `B` ở vị trí 3 không làm thiếu `B` vì còn `B` ở vị trí 9 — đây chính là lý do ta giữ **số đếm**, không chỉ "có/không".
3. **`left` không bao giờ lùi**: `0 → 1 → 6 → 10`. Mỗi ký tự được thêm đúng 1 lần, bỏ tối đa 1 lần.

### Trace với ký tự lặp trong `t`

`s = "BACAAB"`, `t = "AAB"` ⇒ `need = {A:2, B:1}`. Ở đây chữ `A` thứ ba là **thừa**, và khi co ta bỏ được một chữ `A` mà vẫn còn đủ.

In [16]:
trace_sliding("BACAAB", "AAB");

right= 0 +B [0,0] 'B'            A=0/2 B=1/1 formed=1/2  'B' vừa đủ
right= 1 +A [0,1] 'BA'           A=1/2 B=1/1 formed=1/2
right= 2 +C [0,2] 'BAC'          A=1/2 B=1/1 formed=1/2
right= 3 +A [0,3] 'BACA'         A=2/2 B=1/1 formed=2/2  'A' vừa đủ
      co: 'BACA' dài 4 GHI NHẬN; bỏ 'B' -> thiếu 'B', formed=1
right= 4 +A [1,4] 'ACAA'         A=3/2 B=0/1 formed=1/2
right= 5 +B [1,5] 'ACAAB'        A=3/2 B=1/1 formed=2/2  'B' vừa đủ
      co: 'ACAAB' dài 5; bỏ 'A'
      co: 'CAAB' dài 4; bỏ 'C'
      co: 'AAB' dài 3 GHI NHẬN; bỏ 'A' -> thiếu 'A', formed=1
kết quả: 'AAB'


Đi theo từng dòng:

- `right=3`: `A=2/2` **vừa đủ** ⇒ `formed=2/2` ⇒ cửa sổ `"BACA"` (4) được ghi nhận. Bỏ `B` ⇒ thiếu `B` ⇒ dừng co.
- `right=4`: thêm `A` thứ ba ⇒ `A=3/2` thừa ⇒ `formed` vẫn `1/2` (vì đang thiếu `B`).
- `right=5`: thêm `B` ⇒ đủ lại. Co: bỏ `A`@1 ⇒ `A` còn `2/2` **vẫn đủ** ⇒ tiếp tục; `"CAAB"` dài 4 không ngắn hơn; bỏ `C` (rác) ⇒ `"AAB"` dài 3 ⇒ **ghi nhận**; bỏ `A` ⇒ hụt ⇒ dừng.

Kết quả `"AAB"`.

## 11. Cách 3b — Phiên bản gọn với một biến `missing`

Có một cách viết rất phổ biến (hay gặp trong các bài giải "ngắn gọn") chỉ dùng **một** bảng đếm và **một** số nguyên:

- `need = Counter(t)` — nhưng lần này ta **sửa trực tiếp** `need`: `need[c]` = "còn **phải gom thêm** bao nhiêu chữ `c`".
  - `need[c] > 0`: còn thiếu. `need[c] == 0`: vừa đủ. `need[c] < 0`: đang **thừa** `-need[c]` chữ `c` (với ký tự rác, `need` luôn ≤ 0).
- `missing = len(t)` = **tổng số ký tự** (tính cả lặp) còn thiếu. Cửa sổ hợp lệ ⇔ `missing == 0`.

Quy tắc:

- Thêm `c`: nếu `need[c] > 0` (chữ `c` này đang **cần**) ⇒ `missing -= 1`. Sau đó luôn `need[c] -= 1`.
- Khi `missing == 0`: co `left` qua mọi ký tự **thừa** (`need[s[left]] < 0`) — bỏ chúng không làm thiếu gì. Dừng ở ký tự `s[left]` có `need == 0` — bỏ nó là hụt.
- Ghi nhận cửa sổ, rồi **cố ý** bỏ `s[left]` (`need += 1`, `missing += 1`, `left += 1`) để đi tìm cửa sổ tiếp theo.

Ví dụ số với `t = "ABC"`: ban đầu `missing = 3`. Gặp `A` (cần) ⇒ `missing = 2`, `need[A] = 0`. Gặp `D` (rác, `need[D]=0` không > 0) ⇒ `missing` giữ nguyên, `need[D] = -1` (thừa một `D`).

In [17]:
from collections import Counter


def min_window_missing(s: str, t: str) -> str:
    need = Counter(t)            # need[c] = còn phải gom bao nhiêu chữ c (âm = thừa)
    missing = len(t)             # tổng số ký tự còn thiếu
    left = 0
    best_len, best_left = float("inf"), 0

    for right, ch in enumerate(s):
        if need[ch] > 0:         # chữ này đang được cần
            missing -= 1
        need[ch] -= 1

        if missing == 0:
            while need[s[left]] < 0:      # s[left] thừa -> bỏ thoải mái
                need[s[left]] += 1
                left += 1
            if right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            need[s[left]] += 1            # cố ý bỏ s[left] -> lại thiếu 1
            missing += 1
            left += 1

    return "" if best_len == float("inf") else s[best_left:best_left + best_len]


print(min_window_missing("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_missing("BACAAB", "AAB"))          # AAB
print(repr(min_window_missing("a", "aa")))          # ''

BANC
AAB
''


Trace ngắn cho Example 1 — chỉ in các bước có `missing == 0` (lúc tìm được cửa sổ):

In [18]:
from collections import Counter


def trace_missing(s: str, t: str) -> None:
    need = Counter(t)
    missing, left = len(t), 0
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if missing == 0:
            skipped = []
            while need[s[left]] < 0:
                skipped.append(s[left])
                need[s[left]] += 1
                left += 1
            print(f"right={right:>2}: missing=0, bỏ thừa {skipped} -> cửa sổ {s[left:right + 1]!r} dài {right - left + 1}")
            need[s[left]] += 1
            missing += 1
            print(f"          cố ý bỏ '{s[left]}' -> missing={missing}, left={left + 1}")
            left += 1


trace_missing("ADOBECODEBANC", "ABC")

right= 5: missing=0, bỏ thừa [] -> cửa sổ 'ADOBEC' dài 6
          cố ý bỏ 'A' -> missing=1, left=1
right=10: missing=0, bỏ thừa ['D', 'O', 'B', 'E'] -> cửa sổ 'CODEBA' dài 6
          cố ý bỏ 'C' -> missing=1, left=6
right=12: missing=0, bỏ thừa ['O', 'D', 'E'] -> cửa sổ 'BANC' dài 4
          cố ý bỏ 'B' -> missing=1, left=10


Ba cửa sổ "ứng viên" `"ADOBEC"`, `"CODEBA"`, `"BANC"` — giống hệt các cửa sổ ngắn nhất ứng với mỗi lần đạt đủ ở cách 3. Phiên bản `missing` gọn hơn (không cần bảng `window` riêng), nhưng phiên bản `formed` dễ giải thích hơn cho người mới. Bạn nên **hiểu cả hai**, chọn một để thuộc.

Lưu ý: `missing` đếm **tổng số ký tự** (`len(t)`), còn `required` ở cách 3 đếm **số ký tự khác nhau** (`len(Counter(t))`). Trộn lẫn hai định nghĩa là một lỗi hay gặp.

## 12. Cách 4 — Lọc trước: chỉ trượt trên các vị trí có ký tự thuộc `t`

Quan sát: một cửa sổ **ngắn nhất** luôn **bắt đầu và kết thúc** bằng ký tự thuộc `t`. (Nếu nó bắt đầu bằng rác, bỏ ký tự rác đó đi vẫn hợp lệ mà ngắn hơn — mâu thuẫn.) Các ký tự rác chỉ là "đệm" ở giữa.

Vậy nếu `s` dài nhưng chứa ít ký tự thuộc `t` (ví dụ `s` dài `10^5`, `t = "AB"`), ta có thể:

1. Tạo `filtered_s = [(i, s[i]) for i in range(m) if s[i] in need]` — danh sách cặp `(vị trí thật, ký tự)`.
2. Chạy **đúng thuật toán cách 3** trên `filtered_s`; độ dài cửa sổ tính bằng **vị trí thật**: `filtered_s[r][0] - filtered_s[l][0] + 1`.

Độ phức tạp vẫn là `O(m + n)` (vì phải quét `s` một lần để lọc), nhưng vòng hai con trỏ chỉ chạy trên `|filtered_s|` phần tử thay vì `m`. Đây là "Approach 2" trong lời giải chính thức của LeetCode.

Trước hết, xem `filtered_s` trông thế nào:

In [19]:
from collections import Counter

s, t = "ADOBECODEBANC", "ABC"
need = Counter(t)
filtered_s = [(i, ch) for i, ch in enumerate(s) if ch in need]
print(filtered_s)
print(f"m = {len(s)}, |filtered_s| = {len(filtered_s)}")

[(0, 'A'), (3, 'B'), (5, 'C'), (9, 'B'), (10, 'A'), (12, 'C')]
m = 13, |filtered_s| = 6


In [20]:
from collections import Counter


def min_window_filtered(s: str, t: str) -> str:
    need = Counter(t)
    required = len(need)
    filtered_s = [(i, ch) for i, ch in enumerate(s) if ch in need]

    window = Counter()
    formed = 0
    l = 0                                     # chỉ số TRONG filtered_s
    best_len, best_left = float("inf"), 0

    for r in range(len(filtered_s)):
        ch = filtered_s[r][1]
        window[ch] += 1
        if window[ch] == need[ch]:            # ch chắc chắn thuộc t
            formed += 1
        while formed == required:
            start, end = filtered_s[l][0], filtered_s[r][0]   # vị trí THẬT trong s
            if end - start + 1 < best_len:
                best_len, best_left = end - start + 1, start
            out = filtered_s[l][1]
            window[out] -= 1
            if window[out] < need[out]:
                formed -= 1
            l += 1

    return "" if best_len == float("inf") else s[best_left:best_left + best_len]


print(min_window_filtered("ADOBECODEBANC", "ABC"))   # BANC
print(min_window_filtered("BACAAB", "AAB"))          # AAB
print(repr(min_window_filtered("a", "aa")))          # ''

BANC
AAB
''


### Cách 4 lợi bao nhiêu?

Đếm số **bước di chuyển con trỏ** (mỗi lần `right` hoặc `left` tiến một ô) của cách 3 và cách 4 trên một chuỗi dài toàn ký tự "rác", chỉ lác đác vài chữ của `t`. Ta đếm số bước thay vì đo giây để kết quả không phụ thuộc máy.

In [21]:
import random
from collections import Counter


def pointer_moves_sliding(s, t):
    need, window = Counter(t), Counter()
    required, formed, left, moves = len(need), 0, 0, 0
    for right, ch in enumerate(s):
        moves += 1
        window[ch] += 1
        if ch in need and window[ch] == need[ch]:
            formed += 1
        while formed == required:
            out = s[left]
            window[out] -= 1
            if out in need and window[out] < need[out]:
                formed -= 1
            left += 1
            moves += 1
    return moves


def pointer_moves_filtered(s, t):
    need = Counter(t)
    fs = [ch for ch in s if ch in need]       # chỉ cần ký tự để đếm bước
    return pointer_moves_sliding("".join(fs), t)


random.seed(76)
junk = "cdefghijklmnopqrstuvwxyz"
s = "".join(random.choice(junk) if random.random() < 0.98 else random.choice("AB")
            for _ in range(100_000))
t = "AB"
print("m =", len(s), "| số ký tự thuộc t:", sum(ch in "AB" for ch in s))
print("cách 3 - số bước con trỏ:", pointer_moves_sliding(s, t))
print("cách 4 - số bước con trỏ:", pointer_moves_filtered(s, t))
assert min_window_sliding(s, t) == min_window_filtered(s, t)

m = 100000 | số ký tự thuộc t: 2013


cách 3 - số bước con trỏ: 199793


cách 4 - số bước con trỏ: 4023


Khi `s` toàn rác, cách 4 giảm số bước hai con trỏ đi rất nhiều (bù lại tốn `O(m)` bộ nhớ cho `filtered_s` và một lần quét lọc). Khi gần như mọi ký tự của `s` đều thuộc `t`, `filtered_s` dài xấp xỉ `s` và cách 4 **không** lợi gì, thậm chí chậm hơn chút vì phải tạo list. Vì vậy cách 3 vẫn là cách chính nên nộp; cách 4 là tối ưu **hằng số** đáng biết để thảo luận khi phỏng vấn.

## 13. Code nộp LeetCode — Cách 3 (sliding window với `formed`)

Trên LeetCode, `Counter` thường đã được import sẵn; ta vẫn ghi `from collections import Counter` để notebook chạy độc lập và để code tự đầy đủ.

Một chi tiết nhỏ giúp nhanh hơn: nếu `n > m` (t dài hơn s) thì chắc chắn không có đáp án ⇒ trả `""` ngay.

In [22]:
from collections import Counter


class Solution:
    def minWindow(self, s: str, t: str) -> str:
        if len(t) > len(s):
            return ""

        need = Counter(t)
        required = len(need)
        window = Counter()
        formed = 0
        left = 0
        best_len, best_left = float("inf"), 0

        for right, ch in enumerate(s):
            window[ch] += 1
            if ch in need and window[ch] == need[ch]:
                formed += 1

            while formed == required:
                if right - left + 1 < best_len:
                    best_len, best_left = right - left + 1, left
                out = s[left]
                window[out] -= 1
                if out in need and window[out] < need[out]:
                    formed -= 1
                left += 1

        return "" if best_len == float("inf") else s[best_left:best_left + best_len]

### Bộ test

Các nhóm test:

1. 3 ví dụ của đề.
2. `t` dài hơn `s` ⇒ `""`.
3. Phân biệt hoa/thường: `t = "A"` trong `s = "a"` ⇒ `""`.
4. `t` toàn ký tự lặp (`"aa"`, `"aaa"`).
5. Đáp án ở **đầu** chuỗi, ở **cuối** chuỗi, và bằng **cả** chuỗi.
6. Ký tự của `t` nằm rải rác, xa nhau.
7. Có một cửa sổ "gần đúng" nhưng thiếu đúng một bản lặp.

In [23]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS  {actual!r}")


sol = Solution()
check("example 1", sol.minWindow("ADOBECODEBANC", "ABC"), "BANC")
check("example 2", sol.minWindow("a", "a"), "a")
check("example 3", sol.minWindow("a", "aa"), "")
check("t dài hơn s", sol.minWindow("ab", "abc"), "")
check("hoa khác thường", sol.minWindow("a", "A"), "")
check("t lặp 'aa'", sol.minWindow("aa", "aa"), "aa")
check("t lặp 'aaa'", sol.minWindow("baaxaab", "aaa"), "aaxa")
check("đáp án ở đầu", sol.minWindow("abxxxxa", "ab"), "ab")
check("đáp án ở cuối", sol.minWindow("bxxxxab", "ba"), "ab")
check("bằng cả chuỗi", sol.minWindow("cab", "abc"), "cab")
check("rải rác", sol.minWindow("aXXXXXXb", "ba"), "aXXXXXXb")
check("thiếu một bản lặp", sol.minWindow("abcabxy", "aabbcc"), "")
check("hỗn hợp hoa thường", sol.minWindow("aAbBAb", "AB"), "BA")
print("Tất cả test đều PASS!")

example 1                    PASS  'BANC'
example 2                    PASS  'a'
example 3                    PASS  ''
t dài hơn s                  PASS  ''
hoa khác thường              PASS  ''
t lặp 'aa'                   PASS  'aa'
t lặp 'aaa'                  PASS  'aaxa'
đáp án ở đầu                 PASS  'ab'
đáp án ở cuối                PASS  'ab'
bằng cả chuỗi                PASS  'cab'
rải rác                      PASS  'aXXXXXXb'
thiếu một bản lặp            PASS  ''
hỗn hợp hoa thường           PASS  'BA'
Tất cả test đều PASS!


### Test đối chiếu chéo các cách trên input ngẫu nhiên

Sinh `s` và `t` từ bảng chữ cái nhỏ (`"ab"`, `"abc"`, `"aAbB"`) để ký tự hay trùng nhau — dễ có đáp án và dễ lộ bug với ký tự lặp. Giữ `s` ngắn (≤ 12) để brute force `O(m³)` vẫn chạy nhanh. `random.seed(76)` cố định để mỗi lần chạy đều ra cùng dữ liệu.

Vì **mọi** cách ở trên đều trả về cửa sổ ngắn nhất **nằm bên trái nhất** (khi có nhiều cửa sổ cùng độ dài), ta so sánh **chuỗi kết quả** trực tiếp, dù dữ liệu ngẫu nhiên không đảm bảo đáp án duy nhất như đề.

In [24]:
import random

random.seed(76)
non_empty = 0
for _ in range(1500):
    alphabet = random.choice(["ab", "abc", "aAbB"])
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 12)))
    t = "".join(random.choice(alphabet) for _ in range(random.randint(1, 4)))

    a = min_window_bruteforce(s, t)
    b = min_window_fix_left(s, t)
    c = min_window_sliding(s, t)
    d = min_window_missing(s, t)
    e = min_window_filtered(s, t)
    f = Solution().minWindow(s, t)
    assert a == b == c == d == e == f, (s, t, a, b, c, d, e, f)
    non_empty += a != ""

print("1500 bộ test ngẫu nhiên: 6 hàm cho kết quả giống nhau.")
print("Số bộ có đáp án khác rỗng:", non_empty)

1500 bộ test ngẫu nhiên: 6 hàm cho kết quả giống nhau.
Số bộ có đáp án khác rỗng: 946


## 14. Vì sao cách 3 đúng?

### Invariant 1 — `formed` luôn đúng nghĩa

> Ở mọi thời điểm, `window` là bảng đếm **chính xác** của `s[left..right]`, và `formed` = số ký tự `c` của `t` có `window[c] >= need[c]`.

- `window` đúng vì: mỗi lần `right` tiến ta cộng `s[right]`, mỗi lần `left` tiến ta trừ `s[left]` — khớp với việc cửa sổ nhận thêm / mất đi đúng ký tự đó.
- `formed` đúng vì: `window[c]` chỉ thay đổi ±1 mỗi lần, nên trạng thái "`c` đủ / chưa đủ" chỉ đổi **khi đi qua ngưỡng** `need[c]`. Đi lên qua ngưỡng ⇔ sau khi cộng `window[c] == need[c]` ⇒ `+1`. Đi xuống qua ngưỡng ⇔ sau khi trừ `window[c] == need[c] - 1` (tức `< need[c]`) ⇒ `-1`. Các trường hợp khác trạng thái không đổi.

Vậy `formed == required` ⇔ cửa sổ hợp lệ.

### Invariant 2 — không bỏ sót cửa sổ tối ưu

Gọi `[L*, R*]` là cửa sổ tối ưu (ngắn nhất, trái nhất). Ta chứng minh thuật toán chắc chắn ghi nhận nó:

1. **`left` không vượt quá `L*` trước khi `right` tới `R*`.** Để `left` đi từ `L*` sang `L*+1`, vòng `while` phải thấy cửa sổ `[L*, r]` hợp lệ với `r` hiện tại. Nếu `r < R*` thì `[L*, r]` là cửa sổ hợp lệ **ngắn hơn** `[L*, R*]` — mâu thuẫn với tính tối ưu.
2. **Khi `right = R*`**: cửa sổ `[left, R*]` với `left <= L*` chứa trọn `[L*, R*]` ⇒ hợp lệ (tính đơn điệu: thêm ký tự không làm mất hợp lệ). Vòng `while` sẽ co `left` dần; mọi `[p, R*]` với `p <= L*` đều hợp lệ, nên `left` chắc chắn đi tới `L*` và tại đó ghi nhận độ dài `R* - L* + 1`.
3. Mọi cửa sổ được ghi nhận đều hợp lệ (chỉ ghi khi `formed == required`), nên không bao giờ ghi nhầm một cửa sổ thiếu.

Ngoài ra vì ta duyệt `right` tăng dần và chỉ cập nhật khi **ngắn hơn hẳn**, trong các cửa sổ cùng độ dài nhỏ nhất ta giữ cái **xuất hiện sớm nhất** (trái nhất).

## 15. Độ phức tạp

Ký hiệu: `m = len(s)`, `n = len(t)`, `|Σ|` = số ký tự khác nhau có thể có (≤ 52).

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---|---|
| 1. Brute force | Mọi `(i, j)`, đếm lại `Counter` từ đầu | `O(m³ + m²·σ)` | `O(m)` |
| 2. Cố định trái | Mỗi `left` mở rộng `right`, đếm dần, dừng sớm | `O(m²·σ)` | `O(σ)` |
| 3. Sliding window `formed` | Hai con trỏ cùng chiều + `formed` | **`O(m + n)`** | `O(σ)` |
| 3b. Sliding window `missing` | Một bảng `need` có thể âm + `missing` | **`O(m + n)`** | `O(σ)` |
| 4. Lọc `filtered_s` | Cách 3 chỉ trên vị trí có ký tự thuộc `t` | `O(m + n)`, hằng số nhỏ hơn khi `s` nhiều rác | `O(m)` |

(Trong bảng viết `σ` thay cho `|Σ|` để ký tự `|` không làm vỡ bảng markdown.)

Giải thích cách 3:

- `Counter(t)`: `O(n)`.
- `right` đi qua `m` vị trí, `left` cũng đi tối đa `m` vị trí ⇒ tổng `≤ 2m` bước, mỗi bước `O(1)` (cập nhật dict trung bình `O(1)`, so sánh hai số nguyên).
- Cắt chuỗi đáp án một lần cuối: `O(m)`.
⇒ `O(m + n)`. Bộ nhớ: hai bảng đếm, mỗi bảng tối đa `|Σ| = 52` key ⇒ `O(|Σ|)`, coi là `O(1)` khi bảng chữ cái cố định.

Kiểm chứng tốc độ tăng bằng cách đếm số lần "đưa một ký tự vào bảng đếm" của cách 2 và cách 3 (số bước, không phụ thuộc máy):

In [25]:
import random
from collections import Counter


def work_fix_left(s, t):
    need, work = Counter(t), 0
    for left in range(len(s)):
        window = Counter()
        done = False
        for right in range(left, len(s)):
            window[s[right]] += 1
            work += 1
            if covers(window, need):
                done = True
                break
        if not done:
            break
    return work


random.seed(1)
for m in [250, 500, 1000, 2000]:
    s = "".join(random.choice("abcdefghij") for _ in range(m - 1)) + "Z"
    t = "aZ"                     # buộc mọi cửa sổ phải kéo tới tận cuối
    print(f"m = {m:>4}: cách 2 = {work_fix_left(s, t):>9,} bước | cách 3 = {pointer_moves_sliding(s, t):>5,} bước")

m =  250: cách 2 =    31,369 bước | cách 3 =   496 bước


m =  500: cách 2 =   125,244 bước | cách 3 =   996 bước


m = 1000: cách 2 =   500,464 bước | cách 3 = 1,991 bước


m = 2000: cách 2 = 2,000,909 bước | cách 3 = 3,986 bước


Ở ví dụ cố tình xấu này (chữ `Z` duy nhất nằm ở cuối), mỗi `left` của cách 2 đều phải quét tới cuối chuỗi: gấp đôi `m` thì công việc gấp khoảng 4 lần — dấu hiệu `O(m²)`. Cách 3 chỉ tăng gấp đôi — `O(m)`.

## 16. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `set` (hoặc `in`) thay cho bảng đếm

```python
# SAI
if set(t) <= set(window_string): ...
```

`set` bỏ mất số lần lặp: `set("aa") == {"a"}`. Example 3 (`s="a"`, `t="aa"`) sẽ trả `"a"` thay vì `""`.

**Sửa:** luôn so sánh **số đếm**: `window[c] >= need[c]` cho mọi `c`.

In [26]:
def min_window_wrong_set(s: str, t: str) -> str:
    best = ""
    for i in range(len(s)):
        for j in range(i, len(s)):
            if set(t) <= set(s[i:j + 1]):        # SAI: bỏ qua số lần lặp
                if best == "" or j - i + 1 < len(best):
                    best = s[i:j + 1]
    return best


print("set  :", repr(min_window_wrong_set("a", "aa")), "| đúng:", repr(min_window_sliding("a", "aa")))
print("set  :", repr(min_window_wrong_set("BACAAB", "AAB")), "| đúng:", repr(min_window_sliding("BACAAB", "AAB")))

set  : 'a' | đúng: ''
set  : 'BA' | đúng: 'AAB'


### Lỗi 2: Tăng `formed` bằng `>=` thay vì `==`

```python
# SAI
if ch in need and window[ch] >= need[ch]:
    formed += 1
```

Mỗi lần gặp thêm một ký tự **thừa**, `formed` lại bị cộng. Ví dụ `t = "AB"`, `s = "AAB"`: gặp `A` lần 1 ⇒ `formed=1`, gặp `A` lần 2 ⇒ `formed=2 == required` ⇒ tưởng `"AA"` hợp lệ dù không có `B`! Tệ hơn, khi co, bỏ một `A` (`window[A]` từ 2 về 1, chưa "hụt") nên `formed` không giảm ⇒ thuật toán còn ghi nhận luôn `"A"` — output dưới đây trả về `'A'`.

**Sửa:** chỉ cộng khi **vừa đúng bằng** `need[ch]` (đi qua ngưỡng đúng một lần).

In [27]:
from collections import Counter


def sliding_buggy(s, t, use_while=True, ge=False, record_late=False):
    """Bản sliding window có thể bật từng lỗi để quan sát."""
    need, window = Counter(t), Counter()
    required, formed, left = len(need), 0, 0
    best_len, best_left = float("inf"), 0
    for right, ch in enumerate(s):
        window[ch] += 1
        if ch in need and (window[ch] >= need[ch] if ge else window[ch] == need[ch]):
            formed += 1
        while formed == required:
            if not record_late and right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            out = s[left]
            window[out] -= 1
            if out in need and window[out] < need[out]:
                formed -= 1
            left += 1
            if record_late and right - left + 1 < best_len:   # ghi nhận SAU khi đã bỏ s[left]
                best_len, best_left = right - left + 1, left
            if not use_while:
                break                                          # mô phỏng dùng `if` thay `while`
    return "" if best_len == float("inf") else s[best_left:best_left + best_len]


print("lỗi >= :", repr(sliding_buggy("AAB", "AB", ge=True)), "| đúng:", repr(min_window_sliding("AAB", "AB")))
print("lỗi >= :", repr(sliding_buggy("ADOBECODEBANC", "ABC", ge=True)), "| đúng:", repr(min_window_sliding("ADOBECODEBANC", "ABC")))

lỗi >= : 'A' | đúng: 'AB'
lỗi >= : 'BA' | đúng: 'BANC'


### Lỗi 3: Dùng `if` thay cho `while` khi co cửa sổ

```python
# SAI
if formed == required:
    ... ghi nhận, bỏ s[left], left += 1
```

Mỗi lần thêm một ký tự bên phải, cửa sổ chỉ co được **một** bước. Ở Example 1, tại `right=10` cửa sổ `"DOBECODEBA"` cần co 5 bước mới tới `"CODEBA"`, nhưng `if` chỉ co 1 bước ⇒ `left` tụt lại phía sau, bỏ lỡ `"BANC"`.

**Sửa:** `while formed == required:` — co cho tới khi hết hợp lệ.

### Lỗi 4: Ghi nhận đáp án **sau** khi đã bỏ `s[left]`

Thứ tự đúng là: (a) cửa sổ đang hợp lệ ⇒ ghi nhận, (b) rồi mới bỏ `s[left]`. Nếu đảo thứ tự, ta ghi nhận cửa sổ **đã bị hụt** một ký tự — có thể **không hợp lệ** và ngắn hơn đáp án thật.

In [28]:
ex = ("ADOBECODEBANC", "ABC")
print("lỗi if         :", repr(sliding_buggy(*ex, use_while=False)), "| đúng:", repr(min_window_sliding(*ex)))
print("lỗi ghi nhận trễ:", repr(sliding_buggy(*ex, record_late=True)), "| đúng:", repr(min_window_sliding(*ex)))
print("lỗi ghi nhận trễ:", repr(sliding_buggy("a", "a", record_late=True)), "| đúng:", repr(min_window_sliding("a", "a")))

lỗi if         : 'ADOBEC' | đúng: 'BANC'
lỗi ghi nhận trễ: 'ANC' | đúng: 'BANC'
lỗi ghi nhận trễ: '' | đúng: 'a'


`"ANC"` không chứa `B` — đó là cửa sổ đã bị bỏ mất chữ `B` trước khi ghi nhận. Với `s = "a"`, bản ghi nhận trễ còn trả `""` vì nó ghi cửa sổ rỗng độ dài 0 (`right - left + 1 = 0`).

### Lỗi 5: Lệch một khi tính độ dài / cắt chuỗi

- Độ dài cửa sổ `[left, right]` (bao gồm hai đầu) là `right - left + 1`, **không phải** `right - left`.
- Cắt chuỗi: `s[best_left : best_left + best_len]` — vì slicing **không lấy** đầu phải.

### Lỗi 6: Quên trường hợp không có đáp án

Khởi tạo `best_len = float("inf")` và cuối cùng **phải** kiểm tra `best_len == float("inf")` ⇒ trả `""`. Nếu khởi tạo `best_len = 0` hoặc `best_len = len(s)` thì vừa sai khi không có đáp án, vừa sai khi đáp án chính là cả chuỗi `s`.

### Lỗi 7: Nhầm `required` (số ký tự **khác nhau**) với `len(t)` (tổng số ký tự)

Ở cách 3, `formed` đếm số ký tự **khác nhau** đã đủ ⇒ so với `len(Counter(t))`. Nếu so với `len(t)` thì với `t = "AAB"` (`len(t) = 3` nhưng chỉ 2 ký tự khác nhau), `formed` tối đa chỉ đạt 2 ⇒ **không bao giờ** tìm thấy cửa sổ. Ngược lại ở cách 3b, `missing` khởi tạo bằng `len(t)` mới đúng.

### Lỗi 8: Tự ý `.lower()` chuỗi

Đề phân biệt hoa/thường. `minWindow("a", "A")` phải là `""`.

## 17. Mẹo nhận dạng pattern

Nghĩ ngay tới **sliding window kích thước thay đổi** khi đề có đủ các dấu hiệu:

1. Hỏi về **đoạn con liên tiếp** (substring / subarray) — không phải dãy con (subsequence) bỏ cách.
2. Muốn **ngắn nhất** / **dài nhất** / **đếm số đoạn** thỏa một điều kiện.
3. Điều kiện có **tính đơn điệu**: mở rộng đoạn không làm "hỏng" (với bài tìm ngắn nhất), hoặc thu hẹp không làm "hỏng" (với bài tìm dài nhất).
4. Trạng thái của đoạn (tổng, bảng đếm, số ký tự khác nhau, ...) cập nhật được **O(1)** khi thêm/bớt một phần tử ở hai đầu.

Hai khuôn đối xứng nên thuộc lòng:

| Bài tìm **ngắn nhất** (như #76, #209) | Bài tìm **dài nhất** (như #3, #424) |
|---|---|
| `while` cửa sổ **hợp lệ**: ghi nhận, rồi co | `while` cửa sổ **không hợp lệ**: co |
| Ghi nhận **bên trong** vòng `while` | Ghi nhận **sau** vòng `while` |

Ví dụ khuôn "dài nhất" với LeetCode 3 (Longest Substring Without Repeating Characters) — để thấy sự đối xứng:

In [29]:
from collections import Counter


def length_of_longest_substring(s: str) -> int:
    window = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        window[ch] += 1
        while window[ch] > 1:              # KHÔNG hợp lệ (có ký tự lặp) -> co
            window[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)  # ghi nhận SAU khi đã hợp lệ trở lại
    return best


print(length_of_longest_substring("abcabcbb"))   # 3 ("abc")
print(length_of_longest_substring("bbbbb"))      # 1
print(length_of_longest_substring("pwwkew"))     # 3 ("wke")

3
1
3


## 18. Biến thể và bài liên quan

| Bài | Liên hệ với #76 |
|---|---|
| 209. Minimum Size Subarray Sum | Cùng khuôn "ngắn nhất", trạng thái là **tổng** thay cho bảng đếm (mini-demo ở mục 7). |
| 3. Longest Substring Without Repeating Characters | Khuôn đối xứng "dài nhất" (code ngay trên). |
| 567. Permutation in String | Cửa sổ **cố định** độ dài `len(s1)`, hỏi có cửa sổ nào có bảng đếm **bằng đúng** `Counter(s1)` (Bài tập 5). |
| 438. Find All Anagrams in a String | Như 567 nhưng trả về **mọi** vị trí bắt đầu. |
| 30. Substring with Concatenation of All Words | Như 438 nhưng đơn vị là **từ** thay cho ký tự. |
| 424. Longest Repeating Character Replacement | Khuôn "dài nhất" với điều kiện `độ dài - tần suất lớn nhất <= k`. |
| 992. Subarrays with K Different Integers | Đếm số đoạn: `atMost(K) - atMost(K-1)`, mỗi `atMost` là một sliding window. |
| 632. Smallest Range Covering Elements from K Lists | Gộp các list thành dãy `(giá trị, nhãn list)` đã sắp xếp, rồi tìm cửa sổ ngắn nhất **phủ đủ K nhãn** — chính là #76 ngụy trang. |
| 727. Minimum Window Subsequence (Premium) | Yêu cầu `t` là **dãy con có thứ tự** của cửa sổ — mất tính "không cần thứ tự", phải dùng DP hoặc hai con trỏ quét xuôi/ngược. |

Mẹo chuyển đổi: khi gặp bài "tìm đoạn ngắn nhất **phủ đủ** một tập yêu cầu", hãy tự hỏi "`need` là gì? `formed` đếm gì?". Ví dụ bài 632: `need` = mỗi list cần ≥ 1 phần tử, `formed` = số list đã có mặt trong cửa sổ.

## 19. Bài tập vận dụng

Sáu bài tăng dần. Hãy tự làm trước, **đáp án ở cuối notebook** (mục 20). Các cell TODO bên dưới chỉ **định nghĩa** hàm, không gọi — bạn tự bỏ comment các dòng `assert` sau khi làm xong.

### Bài 1 — Đoán output (Counter)

Không chạy code, hãy đoán từng dòng in ra:

```python
from collections import Counter
c = Counter("BANANA")
print(c["A"], c["N"], c["Z"])
c["A"] -= 3
print(c["A"], "A" in c)
need = Counter("AB")
print(need <= Counter("BAN"), need <= Counter("AAN"))
print(len(Counter("AAB")), len("AAB"))
```

Gợi ý mức 1: `Counter` trả `0` cho key chưa có. Gợi ý mức 2: giảm về 0 **không** xóa key. Gợi ý mức 3: `len(Counter)` đếm số **key**.

### Bài 2 — Tính tay bảng trace

Cho `s = "ABAACBAB"`, `t = "ABC"`. Hãy lập bảng như mục 10: mỗi `right` ghi `[left, right]`, `formed`, các bước co (nếu có), và đáp án cuối cùng.

Gợi ý mức 1: `formed` chỉ đạt 3 khi đã có `C`. Gợi ý mức 2: chữ `C` duy nhất ở vị trí 4. Gợi ý mức 3: sau khi co tới khi thiếu `C`, `formed` không thể lên 3 lại nữa.

### Bài 3 — Điền TODO: sliding window với `formed`

Hoàn thiện hàm theo đúng 2 pha "mở rộng – co". Mỗi `TODO` chỉ cần 1–3 dòng.

In [30]:
from collections import Counter


def exercise_min_window(s: str, t: str) -> str:
    need = Counter(t)
    required = len(need)
    window = Counter()
    formed = 0
    left = 0
    best_len, best_left = float("inf"), 0

    for right, ch in enumerate(s):
        # TODO 1: thêm ch vào window; nếu ch "vừa đủ" thì formed += 1
        pass

        # TODO 2: while cửa sổ hợp lệ:
        #   - ghi nhận nếu ngắn hơn best_len
        #   - bỏ s[left] khỏi window; nếu "vừa hụt" thì formed -= 1
        #   - left += 1

    # TODO 3: trả "" nếu chưa từng tìm thấy, ngược lại cắt chuỗi đáp án
    raise NotImplementedError


# Bỏ comment sau khi làm xong:
# assert exercise_min_window("ADOBECODEBANC", "ABC") == "BANC"
# assert exercise_min_window("a", "aa") == ""
# assert exercise_min_window("BACAAB", "AAB") == "AAB"

Gợi ý mức 1: TODO 1 có đúng một phép so sánh `==`. TODO 2 dùng `while formed == required`.

Gợi ý mức 2: trong `while`, thứ tự là **ghi nhận trước, bỏ sau**.

Gợi ý mức 3: `if out in need and window[out] < need[out]: formed -= 1`.

### Bài 4 — Tự viết lại phiên bản `missing` (không nhìn mục 11)

Viết `my_min_window_missing(s, t)` chỉ dùng **một** `Counter` (`need`, được phép âm) và **một** số nguyên `missing = len(t)`.

In [31]:
def my_min_window_missing(s: str, t: str) -> str:
    # TODO: need = Counter(t), missing = len(t)
    # TODO: với mỗi right: nếu need[ch] > 0 thì missing -= 1; need[ch] -= 1
    # TODO: khi missing == 0: bỏ các ký tự thừa ở trái, ghi nhận, rồi cố ý bỏ s[left]
    pass


# Bỏ comment sau khi làm xong:
# assert my_min_window_missing("ADOBECODEBANC", "ABC") == "BANC"
# assert my_min_window_missing("a", "a") == "a"
# assert my_min_window_missing("a", "aa") == ""
# assert my_min_window_missing("aAbBAb", "AB") == "BA"

Gợi ý mức 1: "thừa" nghĩa là `need[s[left]] < 0`.

Gợi ý mức 2: sau vòng `while` bỏ thừa, `s[left]` là ký tự mà bỏ đi sẽ làm thiếu — cửa sổ lúc này là ngắn nhất kết thúc tại `right`.

Gợi ý mức 3: bước "cố ý bỏ" gồm 3 dòng: `need[s[left]] += 1`, `missing += 1`, `left += 1`.

### Bài 5 — Biến thể: LeetCode 567 Permutation in String

Cho `s1`, `s2`. Trả `True` nếu `s2` chứa một **hoán vị** của `s1` làm chuỗi con liên tiếp (tức là có cửa sổ độ dài `len(s1)` có bảng đếm **bằng đúng** `Counter(s1)`).

Ví dụ: `s1 = "ab"`, `s2 = "eidbaooo"` ⇒ `True` (`"ba"`); `s1 = "ab"`, `s2 = "eidboaoo"` ⇒ `False`.

In [32]:
def check_inclusion(s1: str, s2: str) -> bool:
    # TODO: cửa sổ CỐ ĐỊNH độ dài k = len(s1) trượt trên s2
    # TODO: mỗi bước thêm s2[right], nếu cửa sổ dài hơn k thì bỏ s2[right - k]
    # TODO: so sánh bảng đếm cửa sổ với Counter(s1)
    pass


# Bỏ comment sau khi làm xong:
# assert check_inclusion("ab", "eidbaooo") == True
# assert check_inclusion("ab", "eidboaoo") == False
# assert check_inclusion("adc", "dcda") == True
# assert check_inclusion("abc", "ab") == False

Gợi ý mức 1: khác #76 ở chỗ cửa sổ có độ dài **cố định** ⇒ không cần `while`, mỗi bước thêm 1 bỏ 1.

Gợi ý mức 2: so sánh hai `Counter` bằng `==` tốn `O(|Σ|)` mỗi bước — chấp nhận được (`|Σ| = 26`). Muốn `O(1)` thì duy trì `matched` = số ký tự có `window[c] == need[c]`.

Gợi ý mức 3: khi bỏ một ký tự làm đếm về 0, nhớ `del window[c]` nếu bạn so sánh bằng `==` (hoặc dùng Python ≥ 3.10, nơi `Counter` coi key đếm 0 như không có khi so sánh `==`).

### Bài 6 — Mini challenge: cửa sổ ngắn nhất chứa **mọi ký tự khác nhau** của chính `s`

Viết `smallest_distinct_window(s)` trả về chuỗi con ngắn nhất (trái nhất nếu hòa) chứa **tất cả** các ký tự khác nhau xuất hiện trong `s`. Ví dụ `"aabcbcdbca"` có 4 ký tự khác nhau `{a, b, c, d}` ⇒ đáp án `"dbca"`.

Yêu cầu: tự viết sliding window, **không** gọi lại các hàm `min_window_*` ở trên.

In [33]:
def smallest_distinct_window(s: str) -> str:
    # TODO: required = số ký tự khác nhau của s
    # TODO: sliding window: formed = số ký tự khác nhau đang có mặt (đếm >= 1) trong cửa sổ
    pass


# Bỏ comment sau khi làm xong:
# assert smallest_distinct_window("aabcbcdbca") == "dbca"
# assert smallest_distinct_window("aaab") == "ab"
# assert smallest_distinct_window("abc") == "abc"
# assert smallest_distinct_window("a") == "a"

Gợi ý mức 1: đây chính là #76 với `need[c] = 1` cho mọi ký tự khác nhau `c` của `s`.

Gợi ý mức 2: `formed += 1` khi `window[c]` vừa thành `1`; `formed -= 1` khi `window[c]` vừa về `0`.

Gợi ý mức 3: `required = len(set(s))`; phần còn lại giống hệt khuôn mục 9.

## 20. Đáp án tham khảo cho bài tập

Nên tự làm trước rồi mới xem phần này.

### Đáp án bài 1

Chạy trực tiếp để đối chiếu với dự đoán của bạn:

In [34]:
from collections import Counter

c = Counter("BANANA")
print(c["A"], c["N"], c["Z"])                         # 3 2 0
c["A"] -= 3
print(c["A"], "A" in c)                               # 0 True  (key vẫn còn)
need = Counter("AB")
print(need <= Counter("BAN"), need <= Counter("AAN"))  # True False (thiếu B)
print(len(Counter("AAB")), len("AAB"))                # 2 3

3 2 0
0 True
True False
2 3


### Đáp án bài 2

Đáp án cuối là `"ACB"` (vị trí `3..5`). Dùng `trace_sliding` để đối chiếu bảng bạn lập. Các mốc chính:

- `right=4` (thêm `C`): lần đầu `formed = 3/3`. Ghi nhận `"ABAAC"` (5); bỏ `A`@0 vẫn còn `A` ⇒ ghi nhận `"BAAC"` (4); bỏ `B`@1 ⇒ thiếu `B`, `left=2`.
- `right=5` (thêm `B`): đủ lại. `"AACB"` (4) không ngắn hơn; bỏ `A`@2 ⇒ `"ACB"` (3) **ghi nhận**; bỏ `A`@3 ⇒ thiếu `A`, `left=4`.
- `right=6` (thêm `A`): đủ lại. `"CBA"` (3) **không ngắn hơn** 3 nên giữ `"ACB"` (trái hơn); bỏ `C` ⇒ thiếu `C` vĩnh viễn (không còn `C` nào bên phải).

In [35]:
trace_sliding("ABAACBAB", "ABC");

right= 0 +A [0,0] 'A'            A=1/1 B=0/1 C=0/1 formed=1/3  'A' vừa đủ
right= 1 +B [0,1] 'AB'           A=1/1 B=1/1 C=0/1 formed=2/3  'B' vừa đủ
right= 2 +A [0,2] 'ABA'          A=2/1 B=1/1 C=0/1 formed=2/3
right= 3 +A [0,3] 'ABAA'         A=3/1 B=1/1 C=0/1 formed=2/3
right= 4 +C [0,4] 'ABAAC'        A=3/1 B=1/1 C=1/1 formed=3/3  'C' vừa đủ
      co: 'ABAAC' dài 5 GHI NHẬN; bỏ 'A'
      co: 'BAAC' dài 4 GHI NHẬN; bỏ 'B' -> thiếu 'B', formed=2
right= 5 +B [2,5] 'AACB'         A=2/1 B=1/1 C=1/1 formed=3/3  'B' vừa đủ
      co: 'AACB' dài 4; bỏ 'A'
      co: 'ACB' dài 3 GHI NHẬN; bỏ 'A' -> thiếu 'A', formed=2
right= 6 +A [4,6] 'CBA'          A=1/1 B=1/1 C=1/1 formed=3/3  'A' vừa đủ
      co: 'CBA' dài 3; bỏ 'C' -> thiếu 'C', formed=2
right= 7 +B [5,7] 'BAB'          A=1/1 B=2/1 C=0/1 formed=2/3
kết quả: 'ACB'


### Đáp án bài 3 → 6

In [36]:
from collections import Counter


def answer_exercise_min_window(s: str, t: str) -> str:
    need = Counter(t)
    required = len(need)
    window = Counter()
    formed = 0
    left = 0
    best_len, best_left = float("inf"), 0
    for right, ch in enumerate(s):
        window[ch] += 1                                  # TODO 1
        if ch in need and window[ch] == need[ch]:
            formed += 1
        while formed == required:                        # TODO 2
            if right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            out = s[left]
            window[out] -= 1
            if out in need and window[out] < need[out]:
                formed -= 1
            left += 1
    return "" if best_len == float("inf") else s[best_left:best_left + best_len]   # TODO 3


def answer_my_min_window_missing(s: str, t: str) -> str:
    need = Counter(t)
    missing = len(t)
    left = 0
    best_len, best_left = float("inf"), 0
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if missing == 0:
            while need[s[left]] < 0:
                need[s[left]] += 1
                left += 1
            if right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            need[s[left]] += 1
            missing += 1
            left += 1
    return "" if best_len == float("inf") else s[best_left:best_left + best_len]


def answer_check_inclusion(s1: str, s2: str) -> bool:
    k = len(s1)
    if k > len(s2):
        return False
    need = Counter(s1)
    window = Counter()
    for right, ch in enumerate(s2):
        window[ch] += 1
        if right >= k:                       # cửa sổ dài k+1 -> bỏ phần tử bên trái
            out = s2[right - k]
            window[out] -= 1
            if window[out] == 0:
                del window[out]
        if window == need:
            return True
    return False


def answer_smallest_distinct_window(s: str) -> str:
    required = len(set(s))
    window = Counter()
    formed = 0
    left = 0
    best_len, best_left = float("inf"), 0
    for right, ch in enumerate(s):
        window[ch] += 1
        if window[ch] == 1:                  # ký tự mới vừa có mặt
            formed += 1
        while formed == required:
            if right - left + 1 < best_len:
                best_len, best_left = right - left + 1, left
            out = s[left]
            window[out] -= 1
            if window[out] == 0:             # ký tự vừa biến mất khỏi cửa sổ
                formed -= 1
            left += 1
    return s[best_left:best_left + best_len]


assert answer_exercise_min_window("ADOBECODEBANC", "ABC") == "BANC"
assert answer_exercise_min_window("a", "aa") == ""
assert answer_exercise_min_window("BACAAB", "AAB") == "AAB"

assert answer_my_min_window_missing("ADOBECODEBANC", "ABC") == "BANC"
assert answer_my_min_window_missing("a", "a") == "a"
assert answer_my_min_window_missing("a", "aa") == ""
assert answer_my_min_window_missing("aAbBAb", "AB") == "BA"

assert answer_check_inclusion("ab", "eidbaooo") == True
assert answer_check_inclusion("ab", "eidboaoo") == False
assert answer_check_inclusion("adc", "dcda") == True
assert answer_check_inclusion("abc", "ab") == False

assert answer_smallest_distinct_window("aabcbcdbca") == "dbca"
assert answer_smallest_distinct_window("aaab") == "ab"
assert answer_smallest_distinct_window("abc") == "abc"
assert answer_smallest_distinct_window("a") == "a"

# đối chiếu thêm đáp án bài 3, 4 với lời giải chính trên dữ liệu ngẫu nhiên
import random
random.seed(2026)
for _ in range(500):
    s = "".join(random.choice("aAbc") for _ in range(random.randint(1, 15)))
    t = "".join(random.choice("aAbc") for _ in range(random.randint(1, 4)))
    expected = Solution().minWindow(s, t)
    assert answer_exercise_min_window(s, t) == expected
    assert answer_my_min_window_missing(s, t) == expected
    # bài 6 chính là #76 với t = mọi ký tự khác nhau của s
    assert answer_smallest_distinct_window(s) == min_window_sliding(s, "".join(set(s)))

print("Tất cả đáp án đều PASS!")

Tất cả đáp án đều PASS!


## 21. Template cuối cùng nên nhớ

Công thức một dòng: **"Phải mở rộng tới khi đủ, rồi co trái tới khi thiếu; ghi nhận lúc còn đủ."**

```python
from collections import Counter


class Solution:
    def minWindow(self, s: str, t: str) -> str:
        need = Counter(t)
        required = len(need)              # số ký tự KHÁC NHAU
        window = Counter()
        formed = left = 0
        best_len, best_left = float("inf"), 0

        for right, ch in enumerate(s):
            window[ch] += 1                                   # 1) mở rộng
            if ch in need and window[ch] == need[ch]:         #    vừa đủ -> +1
                formed += 1

            while formed == required:                         # 2) còn hợp lệ -> co
                if right - left + 1 < best_len:               #    ghi nhận TRƯỚC
                    best_len, best_left = right - left + 1, left
                out = s[left]                                 #    bỏ SAU
                window[out] -= 1
                if out in need and window[out] < need[out]:   #    vừa hụt -> -1
                    formed -= 1
                left += 1

        return "" if best_len == float("inf") else s[best_left:best_left + best_len]
```

Khuôn tổng quát cho mọi bài "đoạn liên tiếp **ngắn nhất** thỏa điều kiện đơn điệu":

```text
for right in range(n):
    thêm a[right] vào trạng thái
    while trạng thái HỢP LỆ:
        ghi nhận (right - left + 1)
        bỏ a[left] khỏi trạng thái
        left += 1
```

## 22. Checklist tự kiểm tra trước khi submit

- Mình có dùng **bảng đếm** (`Counter`) thay vì `set`/`in` chưa? (ký tự lặp trong `t`, Example 3)
- `required` là `len(Counter(t))` (số ký tự **khác nhau**) chứ không phải `len(t)`? (Nếu dùng bản `missing` thì ngược lại: `missing = len(t)`.)
- Tăng `formed` chỉ khi `window[ch] == need[ch]` (**vừa** đủ), không phải `>=`?
- Giảm `formed` chỉ khi `window[out] < need[out]` (**vừa** hụt) — và chỉ với ký tự thuộc `t`?
- Co cửa sổ bằng **`while`**, không phải `if`?
- Trong `while`: **ghi nhận trước**, rồi mới bỏ `s[left]`?
- Độ dài là `right - left + 1`; cắt đáp án bằng `s[best_left : best_left + best_len]`?
- Khởi tạo `best_len = float("inf")` và trả `""` khi không tìm thấy?
- Không `.lower()` — chữ hoa và chữ thường khác nhau?
- Đã thử: 3 ví dụ đề, `t` dài hơn `s`, `t` toàn ký tự lặp, đáp án ở đầu / cuối / bằng cả `s`, hoa-thường lẫn lộn?
- Độ phức tạp `O(m + n)`: mỗi con trỏ chỉ tiến, không bao giờ lùi?